# 06.1 — Pandas Series

## Learning Objectives

By the end of this notebook you should be able to:

- Create a `Series` from a Python list and a dictionary
- Explain what the **index** is and why it makes a Series different from a list
- Access elements with `[]`, `.loc[]`, and `.iloc[]`
- Perform vectorized arithmetic and comparison operations on a Series
- Filter a Series using boolean indexing
- Use common methods: `.describe()`, `.value_counts()`, `.sort_values()`, `.unique()`, `.isnull()`
- Use the `.str` accessor to work with string data

In [ ]:
import pandas as pd

## 1. What Is a Series?

A **Series** is pandas' one-dimensional data structure — a sequence of values, each paired with a label called the **index**.

On the surface a Series looks like a Python list. The key difference is that every value has a *named label*, not just a position number. Think of it as a list and a dictionary combined:

- Like a **list**: values are ordered; you can access them by position.
- Like a **dict**: every value has a name (the index); you can look up values by label.

| | Python list | Python dict | pandas Series |
|---|---|---|---|
| Ordered? | ✓ | insertion order | ✓ |
| Named labels? | ✗ | ✓ | ✓ |
| Vectorized math? | ✗ | ✗ | ✓ |
| Handles missing values? | ✗ | ✗ | ✓ |

In [ ]:
# Create a Series from a Python list
# Pandas assigns a default integer index starting at 0
ages = pd.Series([22, 38, 26, 35, 28])
ages

The output shows two columns: the **index** on the left and the **values** on the right. The `dtype: int64` line tells you pandas is storing these as 64-bit integers.

The index and values are separate attributes:

In [ ]:
print("Index:  ", ages.index)
print("Values: ", ages.values)
print("dtype:  ", ages.dtype)

## 2. Creating a Series with a Meaningful Index

An integer index is fine for ordered data, but often you want each value to have a human-readable label.

In [ ]:
# Assign a custom index
ages = pd.Series(
    [22, 38, 26, 35, 28],
    index=["Alice", "Bob", "Carol", "Dan", "Eve"]
)
ages

In [ ]:
# Creating from a dictionary: keys become the index, values become the data
scores = pd.Series({
    "math":    91,
    "english": 84,
    "science": 78,
    "history": 95,
})
scores

## 3. Accessing Elements

| Syntax | Selects by | Best used when |
|---|---|---|
| `s["label"]` | label | index has meaningful names |
| `s.loc["label"]` | label (explicit, always safe) | preferred form |
| `s.iloc[2]` | integer position | you care about order, not labels |

In [ ]:
# .loc[] — label-based access
print(ages.loc["Bob"])        # 38
print(scores.loc["science"])  # 78

In [ ]:
# .iloc[] — position-based (0-indexed, like Python lists)
print(ages.iloc[0])    # first element: 22
print(ages.iloc[-1])   # last element:  28

In [ ]:
# Slicing
# iloc: end is EXCLUSIVE (Python convention)
print(ages.iloc[1:3])
print()
# loc: end is INCLUSIVE (label-based convention)
print(ages.loc["Bob":"Dan"])

## 4. Vectorized Operations

Apply arithmetic or comparisons to an *entire Series* in one expression — no loop needed. Pandas applies the operation element-by-element and returns a new Series with the same index.

In [ ]:
fares = pd.Series(
    [7.25, 71.28, 7.92, 53.10, 8.05],
    index=["Alice", "Bob", "Carol", "Dan", "Eve"]
)

# Scalar arithmetic: applied to every element
fares_in_pounds = fares * 0.79
fares_in_pounds

In [ ]:
# Arithmetic between two Series: pandas aligns on the index
discount = pd.Series(
    [2.00, 0.00, 1.00, 5.00, 0.00],
    index=["Alice", "Bob", "Carol", "Dan", "Eve"]
)

final_price = fares - discount
final_price

## 5. Boolean Indexing (Filtering)

A comparison like `> 25` returns a **boolean Series** — `True` where the condition holds, `False` elsewhere. Pass that boolean Series into `[]` or `.loc[]` to keep only matching rows.

In [ ]:
# Step 1: create the mask
mask = ages > 25
print(mask)

In [ ]:
# Step 2: apply the mask
ages[mask]

In [ ]:
# Inline — same result
ages[ages > 25]

In [ ]:
# Combine conditions with & (AND) and | (OR)
# Each condition MUST be in its own parentheses
ages[(ages >= 25) & (ages <= 35)]

> **Common mistake:** Use `&` and `|`, **not** Python's `and` / `or` keywords — they don't work element-by-element on a Series.

## 6. Useful Series Methods

In [ ]:
# .describe() — statistical summary in one call
fares.describe()

In [ ]:
# .value_counts() — frequency of each unique value; great for categorical data
pclass = pd.Series([3, 1, 3, 1, 3, 2, 3, 3, 2, 2])
pclass.value_counts()

In [ ]:
# .sort_values() returns a new sorted Series; the original is unchanged
print(fares.sort_values())                 # ascending (default)
print()
print(fares.sort_values(ascending=False))  # descending

In [ ]:
# .unique() and .nunique()
print("Distinct classes:", pclass.unique())
print("How many distinct:", pclass.nunique())

## 7. Missing Values

Real-world data often has gaps. Pandas represents missing numeric values as `NaN` (Not a Number). `.isnull()` returns `True` where data is missing; chaining `.sum()` counts the missing entries.

In [ ]:
# Create a Series with missing values (None becomes NaN for numeric data)
heights = pd.Series(
    [175.0, None, 162.0, None, 180.0],
    index=["Alice", "Bob", "Carol", "Dan", "Eve"]
)

print(heights.isnull())
print()
print("Missing count:", heights.isnull().sum())

## 8. The `.str` Accessor

When a Series holds strings, the `.str` accessor unlocks string methods that work element-by-element. Common: `.str.lower()`, `.str.strip()`, `.str.contains()`, `.str.len()`, `.str.replace()`.

In [ ]:
names = pd.Series(["Alice Smith", "  Bob Jones ", "CAROL LEE", "dan brown"])

print(names.str.lower().str.strip())   # normalize case and whitespace
print()
print(names.str.lower().str.contains("lee"))  # boolean mask
print()
print(names.str.len())                 # length of each string

## 9. Series from a Real Dataset

In practice you will rarely construct a Series by hand. Almost always, a Series is a **column extracted from a DataFrame**. The next notebook covers DataFrames fully — here is a preview.

In [ ]:
url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

In [ ]:
# A single column of a DataFrame is a Series
age_series = df["age"]
print(type(age_series))
age_series.head(10)

In [ ]:
print("Total rows:  ", len(age_series))
print("Missing ages:", age_series.isnull().sum())  # 0 — this dataset is complete
print()
age_series.describe()

In [ ]:
# Boolean filter: passengers older than 60
seniors = age_series[age_series > 60]
print(f"Passengers over 60: {len(seniors)}")
seniors.head()

## Your Turn

Use the `df["fare"]` column (ticket prices) for the following.

1. What is the most expensive fare? The cheapest?  *(Hint: `.max()`, `.min()`)*
2. How many passengers paid more than £50?  *(Hint: boolean mask + `.sum()`)*
3. What is the median fare?  *(Hint: `.median()`)*
4. Sort fares from lowest to highest and display the five cheapest.  *(Hint: `.sort_values().head()`)*
5. What fraction of passengers paid less than £10?  *(Hint: boolean mask → `.mean()` on the result gives the proportion)*

In [ ]:
fare_series = df["fare"]

# Your code here